In [ ]:
import sys; from pathlib import Path; sys.path.insert(0, str(Path.home() / "Github" / "the-bible-catalog"))

In [ ]:
from config.settings import *
from config._common_libraries import *
from config._common_functions import *
from config._util_functions import *

In [ ]:
import os
import re
import json
import time
import requests
import pandas as pd
from datetime import datetime
from pathlib import Path

# ─────────────────────────────────────────────
# Bible Canon — 66 Books
# ─────────────────────────────────────────────

BIBLE_CANON = [
    # ── Old Testament (39) ──────────────────
    ("Genesis",         "Old Testament", 50),
    ("Exodus",          "Old Testament", 40),
    ("Leviticus",       "Old Testament", 27),
    ("Numbers",         "Old Testament", 36),
    ("Deuteronomy",     "Old Testament", 34),
    ("Joshua",          "Old Testament", 24),
    ("Judges",          "Old Testament", 21),
    ("Ruth",            "Old Testament",  4),
    ("1 Samuel",        "Old Testament", 31),
    ("2 Samuel",        "Old Testament", 24),
    ("1 Kings",         "Old Testament", 22),
    ("2 Kings",         "Old Testament", 25),
    ("1 Chronicles",    "Old Testament", 29),
    ("2 Chronicles",    "Old Testament", 36),
    ("Ezra",            "Old Testament", 10),
    ("Nehemiah",        "Old Testament", 13),
    ("Esther",          "Old Testament", 10),
    ("Job",             "Old Testament", 42),
    ("Psalms",          "Old Testament",150),
    ("Proverbs",        "Old Testament", 31),
    ("Ecclesiastes",    "Old Testament", 12),
    ("Song of Solomon", "Old Testament",  8),
    ("Isaiah",          "Old Testament", 66),
    ("Jeremiah",        "Old Testament", 52),
    ("Lamentations",    "Old Testament",  5),
    ("Ezekiel",         "Old Testament", 48),
    ("Daniel",          "Old Testament", 12),
    ("Hosea",           "Old Testament", 14),
    ("Joel",            "Old Testament",  3),
    ("Amos",            "Old Testament",  9),
    ("Obadiah",         "Old Testament",  1),
    ("Jonah",           "Old Testament",  4),
    ("Micah",           "Old Testament",  7),
    ("Nahum",           "Old Testament",  3),
    ("Habakkuk",        "Old Testament",  3),
    ("Zephaniah",       "Old Testament",  3),
    ("Haggai",          "Old Testament",  2),
    ("Zechariah",       "Old Testament", 14),
    ("Malachi",         "Old Testament",  4),
    # ── New Testament (27) ──────────────────
    ("Matthew",         "New Testament", 28),
    ("Mark",            "New Testament", 16),
    ("Luke",            "New Testament", 24),
    ("John",            "New Testament", 21),
    ("Acts",            "New Testament", 28),
    ("Romans",          "New Testament", 16),
    ("1 Corinthians",   "New Testament", 16),
    ("2 Corinthians",   "New Testament", 13),
    ("Galatians",       "New Testament",  6),
    ("Ephesians",       "New Testament",  6),
    ("Philippians",     "New Testament",  4),
    ("Colossians",      "New Testament",  4),
    ("1 Thessalonians", "New Testament",  5),
    ("2 Thessalonians", "New Testament",  3),
    ("1 Timothy",       "New Testament",  6),
    ("2 Timothy",       "New Testament",  4),
    ("Titus",           "New Testament",  3),
    ("Philemon",        "New Testament",  1),
    ("Hebrews",         "New Testament", 13),
    ("James",           "New Testament",  5),
    ("1 Peter",         "New Testament",  5),
    ("2 Peter",         "New Testament",  3),
    ("1 John",          "New Testament",  5),
    ("2 John",          "New Testament",  1),
    ("3 John",          "New Testament",  1),
    ("Jude",            "New Testament",  1),
    ("Revelation",      "New Testament", 22),
]

# Lookup: book_name → (testament, chapter_count)
BOOK_LOOKUP = {book: (testament, chapters) for book, testament, chapters in BIBLE_CANON}


# ─────────────────────────────────────────────
# ESV API Parameters
# Tuned for clean AI/NLP processing
# ─────────────────────────────────────────────

ESV_DEFAULT_PARAMS = {
    "include-passage-references":  False,  # Tracked in schema
    "include-verse-numbers":       True,   # Required for verse parsing
    "include-first-verse-numbers": True,
    "include-footnotes":           False,  # Clean text for AI
    "include-footnote-body":       False,
    "include-headings":            False,  # Clean text for AI
    "include-short-copyright":     False,  # Added at display layer
    "include-selahs":              True,   # Preserve Psalm poetic markers
    "indent-using":                "space",
    "line-length":                 0,      # No artificial line wrapping
}


# ─────────────────────────────────────────────
# ESVBibleIngestion Class
# ─────────────────────────────────────────────

class ESVBibleIngestion:
    """
    ESV Bible ingestion client for TrioTheo Bronze layer.

    Fetches ESV passage text from api.esv.org and returns
    structured pandas DataFrames matching the TrioTheo schema.

    Schema:
        translation  (str)  : "ESV"
        testament    (str)  : "Old Testament" | "New Testament"
        book         (str)  : Full book name
        chapter      (int)  : Chapter number
        verse_number (int)  : Verse number
        verse_text   (str)  : Clean verse text
    """

    BASE_URL      = "https://api.esv.org/v3/passage/text/"
    RATE_LIMIT    = 0.25   # seconds between requests
    MAX_RETRIES   = 3
    RETRY_BACKOFF = 2.0

    def __init__(self, api_key: str = None, rate_limit: float = None):
        """
        Args:
            api_key:    ESV API token. Falls back to ESV_API_KEY env var.
            rate_limit: Override default request delay (seconds).
        """
        self.api_key = api_key or os.environ.get("ESV_API_KEY", "")
        if not self.api_key:
            raise EnvironmentError(
                "ESV API key required. Pass api_key= or set ESV_API_KEY env var."
            )
        self.rate_limit = rate_limit or self.RATE_LIMIT
        self._headers  = {"Authorization": f"Token {self.api_key}"}

    # ── Public Methods ───────────────────────────────────────────

    def get_passage_df(self, passage: str) -> pd.DataFrame:
        """
        Fetch any passage reference and return a verse-per-row DataFrame.

        Args:
            passage: Any ESV-compatible reference.
                     e.g. "Genesis 1-3", "John 3:16", "Romans 8:1-4"

        Returns:
            pd.DataFrame with TrioTheo schema columns.

        Example:
            df = ingestor.get_passage_df("Genesis 1-3")
        """
        raw = self._fetch_raw(passage)
        passages = raw.get("passages", [])
        if not passages:
            print(f"  ⚠ No passage data returned for: {passage!r}")
            return self._empty_df()

        # Detect book/testament from canonical reference
        canonical  = raw.get("canonical", passage)
        book, testament = self._resolve_book_testament(canonical, passage)

        rows = self._parse_passage_text(passages[0], book, testament)
        return self._to_df(rows)

    def get_book_df(self, book_name: str) -> pd.DataFrame:
        """
        Fetch all chapters of a single book.

        Args:
            book_name: Full book name e.g. "Romans", "1 Corinthians"

        Returns:
            pd.DataFrame with all verses in that book.

        Example:
            df = ingestor.get_book_df("Romans")
        """
        if book_name not in BOOK_LOOKUP:
            raise ValueError(f"Unknown book: {book_name!r}. Check BIBLE_CANON for valid names.")

        testament, chapter_count = BOOK_LOOKUP[book_name]
        all_rows = []

        for chapter_num in range(1, chapter_count + 1):
            ref = f"{book_name} {chapter_num}"
            raw = self._fetch_raw(ref)
            passages = raw.get("passages", [])

            if passages:
                rows = self._parse_passage_text(passages[0], book_name, testament)
                all_rows.extend(rows)
            else:
                print(f"  ⚠ No data for {ref}")

            time.sleep(self.rate_limit)

        return self._to_df(all_rows)

    def get_full_bible_df(
        self,
        save_parquet: str = None,
        save_csv:     str = None,
        progress:     bool = True,
    ) -> pd.DataFrame:
        """
        Ingest all 66 books of the ESV Bible (~7 min at default rate limit).

        Args:
            save_parquet: Optional file path to save .parquet output.
            save_csv:     Optional file path to save .csv output.
            progress:     Print progress updates (default True).

        Returns:
            pd.DataFrame with ~31,102 verse rows.

        Example:
            df = ingestor.get_full_bible_df(save_parquet="esv_bible.parquet")
        """
        all_rows   = []
        audit_log  = []
        start_time = datetime.now()

        if progress:
            print(f"Starting ESV full Bible ingestion — {start_time.strftime('%Y-%m-%d %H:%M:%S %Z')}")
            print(f"Target: 66 books, 1,189 chapters, ~31,102 verses\n")

        for book_idx, (book_name, testament, chapter_count) in enumerate(BIBLE_CANON, 1):
            book_rows = []

            if progress:
                print(f"[{book_idx:02d}/66] {book_name} ({chapter_count} ch)...", end=" ", flush=True)

            for chapter_num in range(1, chapter_count + 1):
                ref = f"{book_name} {chapter_num}"
                raw = self._fetch_raw(ref)
                passages = raw.get("passages", [])

                if passages:
                    rows = self._parse_passage_text(passages[0], book_name, testament)
                    book_rows.extend(rows)
                    audit_log.append({"book": book_name, "chapter": chapter_num,
                                      "status": "OK", "verse_count": len(rows)})
                else:
                    print(f"  ⚠ Empty response: {ref}")
                    audit_log.append({"book": book_name, "chapter": chapter_num,
                                      "status": "EMPTY", "verse_count": 0})

                time.sleep(self.rate_limit)

            all_rows.extend(book_rows)

            if progress:
                print(f"✓ {len(book_rows)} verses")

        df = self._to_df(all_rows)
        end_time = datetime.now()
        duration = (end_time - start_time).total_seconds()

        if progress:
            self._print_validation(df, duration)

        if save_parquet:
            df.to_parquet(save_parquet, index=False, engine="pyarrow", compression="snappy")
            print(f"\nSaved → {save_parquet}")

        if save_csv:
            df.to_csv(save_csv, index=False, encoding="utf-8")
            print(f"Saved → {save_csv}")

        self._save_audit(audit_log, start_time, end_time, df)

        return df

    # ── Private: API ─────────────────────────────────────────────

    def _fetch_raw(self, passage: str) -> dict:
        """Fetch raw API response with retry logic."""
        params = {"q": passage, **ESV_DEFAULT_PARAMS}

        for attempt in range(1, self.MAX_RETRIES + 1):
            try:
                resp = requests.get(
                    self.BASE_URL, headers=self._headers,
                    params=params, timeout=15
                )
                resp.raise_for_status()
                return resp.json()

            except requests.exceptions.HTTPError as e:
                code = e.response.status_code
                if code == 400:
                    print(f"  ⚠ Bad request — skipping: {passage!r}")
                    return {}

            except requests.exceptions.RequestException as e:
                print(f"  ⚠ Request error on {passage!r}, attempt {attempt}: {e}")

            if attempt < self.MAX_RETRIES:
                time.sleep(self.rate_limit * (self.RETRY_BACKOFF ** attempt))

        print(f"  ✗ All {self.MAX_RETRIES} attempts failed for {passage!r}")
        return {}

    # ── Private: Parsing ─────────────────────────────────────────

    def _parse_passage_text(
        self, passage_text: str, book: str, testament: str
    ) -> list[dict]:
        """
        Parse verse-per-row records from ESV API passage text.

        The ESV API returns text with verse markers like [1], [12].
        This method splits on those markers to isolate each verse.

        Handles:
          - Prose verses
          - Poetry / indented lines (Psalms, Proverbs)
          - Selah markers
          - Em dashes and special punctuation
          - Multi-line verse continuations
        """
        if not passage_text:
            return []

        timestamp = datetime.now().isoformat()

        # Split on [verse_number] markers — capturing group keeps the number
        parts = re.split(r'\[(\d+)\]', passage_text)

        # parts layout: [pre_text, verse_num, verse_text, verse_num, verse_text, ...]
        rows = []
        i = 1
        while i < len(parts) - 1:
            verse_num_str = parts[i].strip()
            raw_text      = parts[i + 1]

            # Normalize whitespace while preserving content
            # Collapse internal newlines/extra spaces → single space
            verse_text = re.sub(r'[ \t]+', ' ', raw_text)       # collapse spaces/tabs
            verse_text = re.sub(r'\n+', ' ', verse_text)         # collapse newlines
            verse_text = verse_text.strip()

            # Remove trailing section headers (all-caps lines sometimes bleed in)
            verse_text = re.sub(r'\s{2,}[A-Z][A-Z\s,]+$', '', verse_text).strip()

            if verse_num_str.isdigit() and verse_text:
                rows.append({
                    "translation":  "ESV",
                    "testament":    testament,
                    "book":         book,
                    "chapter":      None,
                    "verse_number": int(verse_num_str),
                    "verse_text":   verse_text,
                })
            i += 2

        # ── Resolve chapter numbers ──────────────────────────────
        # When fetching multi-chapter passages (e.g. "Genesis 1-3"),
        # the API returns all chapters concatenated. We detect chapter
        # boundaries by watching for verse_number resets (back to 1).
        if rows:
            # Determine starting chapter from book lookup if possible
            _, chapter_count = BOOK_LOOKUP.get(book, ("Unknown", 1))
            chapter = self._detect_starting_chapter(book, rows)

            for idx, row in enumerate(rows):
                if idx > 0 and row["verse_number"] == 1:
                    chapter += 1
                row["chapter"] = chapter

        return rows

    def _detect_starting_chapter(self, book: str, rows: list[dict]) -> int:
        """
        Best-effort detection of the starting chapter.
        Defaults to 1 for single-chapter requests.
        """
        # For single-book full ingestion calls, chapter is always tracked by caller.
        # For ad-hoc passage calls, default to 1 (works for most use cases).
        return 1

    def _resolve_book_testament(self, canonical: str, fallback: str) -> tuple[str, str]:
        """Extract book name and testament from a canonical reference string."""
        # Try to match canonical ref like "Genesis 1:1" or "1 Corinthians 13:4–7"
        for book_name in BOOK_LOOKUP:
            if canonical.startswith(book_name):
                testament, _ = BOOK_LOOKUP[book_name]
                return book_name, testament

        # Fallback: parse the passage query itself
        for book_name in BOOK_LOOKUP:
            if fallback.startswith(book_name):
                testament, _ = BOOK_LOOKUP[book_name]
                return book_name, testament

        return "Unknown", "Unknown"

    # ── Private: DataFrame ───────────────────────────────────────

    def _to_df(self, rows: list[dict]) -> pd.DataFrame:
        """Convert row list to typed DataFrame."""
        if not rows:
            return self._empty_df()

        df = pd.DataFrame(rows, columns=[
            "translation",
            "testament",
            "book",
            "chapter",
            "verse_number",
            "verse_text",
        ])

        df["chapter"]      = df["chapter"].astype("Int16")
        df["verse_number"] = df["verse_number"].astype("Int16")
        df["translation"]  = df["translation"].astype("category")
        df["testament"]    = df["testament"].astype("category")
        df["book"]         = df["book"].astype("string")
        df["verse_text"]   = df["verse_text"].astype("string")

        return df.reset_index(drop=True)

    def _empty_df(self) -> pd.DataFrame:
        """Return an empty DataFrame with the correct schema."""
        return pd.DataFrame(columns=[
            "translation", "testament", "book", "chapter",
            "verse_number", "verse_text"
        ])

    # ── Private: Reporting ───────────────────────────────────────

    def _print_validation(self, df: pd.DataFrame, duration: float):
        delta = len(df) - 31_102
        print(f"\n{'='*50}")
        print(f"  INGESTION COMPLETE")
        print(f"{'='*50}")
        print(f"  Total verses   : {len(df):,}  (expected 31,102 | delta {delta:+,})")
        print(f"  Books          : {df['book'].nunique()} / 66")
        print(f"  OT verses      : {len(df[df['testament'] == 'Old Testament']):,}")
        print(f"  NT verses      : {len(df[df['testament'] == 'New Testament']):,}")
        print(f"  Null text rows : {df['verse_text'].isna().sum()}")
        print(f"  Duration       : {duration:.0f}s ({duration/60:.1f} min)")
        print(f"{'='*50}")

    def _save_audit(self, audit_log, start_time, end_time, df):
        """Save chapter-level audit log for traceability."""
        audit = {
            "ingest_start":     start_time.isoformat(),
            "ingest_end":       end_time.isoformat(),
            "duration_seconds": (end_time - start_time).total_seconds(),
            "total_verses":     len(df),
            "books_ingested":   df["book"].nunique(),
            "chapters":         audit_log,
        }
        audit_path = Path("esv_ingest_audit.json")
        with open(audit_path, "w") as f:
            json.dump(audit, f, indent=2)

In [ ]:
ingestor = ESVBibleIngestion(api_key=EXTERNAL_SERVICES["esv_api_token"])

In [ ]:
df = ingestor.get_book_df("Philemon")        # 1 call, full tiny book

In [ ]:
display(df)

In [ ]:
import requests
import os

ESV_API_TOKEN = EXTERNAL_SERVICES["esv_api_token"]
ESV_BASE_URL = "https://api.esv.org/v3/passage/text/"

HEADERS = {
    "Authorization": f"Token {ESV_API_TOKEN}"
}

def fetch_esv_passage(passage: str, **kwargs) -> dict:
    """
    Fetch ESV passage text from the API.
    
    Args:
        passage: Passage reference e.g. "John 3:16", "Genesis 1-3"
        **kwargs: Optional API parameters (see below)
    
    Returns:
        dict with keys: query, canonical, passage_meta, passages
    """
    params = {
        "q": passage,
        "include-headings": True,
        "include-footnotes": False,       # Clean text for AI processing
        "include-footnote-body": False,
        "include-short-copyright": True,  # Required for ESV license compliance
        "include-verse-numbers": True,
        **kwargs
    }
    
    response = requests.get(ESV_BASE_URL, headers=HEADERS, params=params)
    response.raise_for_status()
    return response.json()


def get_passage_text(passage: str) -> str:
    """Returns clean passage text string only."""
    data = fetch_esv_passage(passage)
    passages = data.get("passages", [])
    return passages[0].strip() if passages else ""

In [ ]:
text = get_passage_text("Genesis 1-3")
print(text)